# Step 1 — Install Dependencies


In [1]:
!pip install langchain openai faiss-cpu tiktoken pymupdf



   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 12.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/785.8 kB ? eta -:--:--
   --------------------------------------- 785.8/785.8 kB 16.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ------------------------ --------------- 9.2/14.9 MB 43.9 MB/s eta 0:00:01
   ---------------------------------------  14.7/14.9 MB 44.0 MB/s eta 0:00:01
   ---------------------------------------- 14.9/14.9 MB 34.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/875.7 kB ? eta -:--:--
   --------------------------------------- 875.7/875.7 kB 19.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
   ------------------- -------------------- 9.2/18.7 MB 43.9 MB/s eta 0:00:01
   -------------------------------------- - 17.8/18.7 MB 41.6 MB/s eta 0:00:01
   ----------------

In [10]:
!pip install -U langchain-community


  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached typing_extensions-4.14.1-py3-none-any.whl.metadata (3.0 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 24.1 MB/s eta 0:00:00
Using cached typing_inspection-0.4.1-py3-none-any.whl (14 kB)
Using cached typing_extensions-4.14.1-py3-none-any.whl (43 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0


# Step 2 — Set API Key and Model

In [2]:
import openai

# Paste your OpenAI API key 
openai.api_key = "YOUR_API_KEY_HERE"

# Model
MODEL_NAME = "gpt-3.5-turbo"


# Step 3 — Load PDFs from Folder

In [7]:
import fitz  # PyMuPDF
import os

def load_pdfs_from_folder(folder_path):
    all_texts = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            pdf_doc = fitz.open(file_path)
            text = ""
            for page in pdf_doc:
                text += page.get_text()
            all_texts.append({"filename": filename, "text": text})
    return all_texts

# Load all PDFs from docs folder
pdf_texts = load_pdfs_from_folder("docs")

# Show first 500 characters of the first PDF
if pdf_texts:
    print(f"Loaded {len(pdf_texts)} PDFs.")
    print(f"Example from: {pdf_texts[0]['filename']}\n")
    print(pdf_texts[0]['text'][:500])
else:
    print("No PDFs found in docs/ folder.")


Loaded 1 PDFs.
Example from: sample.pdf

Process for Adapting Language Models to
Society (PALMS) with Values-Targeted
Datasets
Irene Solaiman∗
Zillow Group
contact@irenesolaiman.com
Christy Dennison∗
MIT
christy@mit.edu
Abstract
Language models can generate harmful and biased outputs and exhibit un-
desirable behavior according to a given cultural context.
We propose a
Process for Adapting Language Models to Society (PALMS) with Values-
Targeted Datasets, an iterative process to signiﬁcantly change model behav-
ior by crafting and ﬁne-


# Step 4 — Chunk Text

In [8]:
from langchain.text_splitter import CharacterTextSplitter

# Combine all PDFs into one big text list
all_text = ""
for pdf in pdf_texts:
    all_text += pdf["text"] + "\n"

# Create the text splitter
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,     # each chunk ~1000 characters
    chunk_overlap=150,   # overlap between chunks
    length_function=len
)

# Split into chunks
chunks = text_splitter.split_text(all_text)

print(f"Total chunks created: {len(chunks)}")
print("\nFirst chunk preview:\n")
print(chunks[0])


Total chunks created: 116

First chunk preview:

Process for Adapting Language Models to
Society (PALMS) with Values-Targeted
Datasets
Irene Solaiman∗
Zillow Group
contact@irenesolaiman.com
Christy Dennison∗
MIT
christy@mit.edu
Abstract
Language models can generate harmful and biased outputs and exhibit un-
desirable behavior according to a given cultural context.
We propose a
Process for Adapting Language Models to Society (PALMS) with Values-
Targeted Datasets, an iterative process to signiﬁcantly change model behav-
ior by crafting and ﬁne-tuning on a dataset that reﬂects a predetermined set
of target values. We evaluate our process using three metrics: quantitative
metrics with human evaluations that score output adherence to a target
value, toxicity scoring on outputs; and qualitative metrics analyzing the
most common word associated with a given social category. Through each
iteration, we add additional training dataset examples based on observed
shortcomings from evaluations. PA

# Step 5 — Create Embeddings and Store in FAISS

In [11]:
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


# Create embeddings object
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=openai.api_key)

# Create FAISS vector store from chunks
vectorstore = FAISS.from_texts(chunks, embeddings)

# Save the FAISS index locally
vectorstore.save_local("faiss_index")

print(" Embeddings created and FAISS index built!")


C:\Users\HP\AppData\Local\Temp\ipykernel_20172\3874961411.py:6: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=openai.api_key)


 Embeddings created and FAISS index built!


In [12]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

# Create retriever from FAISS index
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# Create the GPT model
llm = ChatOpenAI(model_name=MODEL_NAME, openai_api_key=openai.api_key)

# Create RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

# Ask a question
query = "What is the PALMS process about?"
result = qa_chain({"query": query})

# Print the answer
print("Answer:\n", result["result"])

# Show sources
print("\nSources:")
for doc in result["source_documents"]:
    print(f"- {doc.page_content[:200]}...")  # show first 200 chars of each source


C:\Users\HP\AppData\Local\Temp\ipykernel_20172\3917012578.py:8: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model_name=MODEL_NAME, openai_api_key=openai.api_key)
C:\Users\HP\AppData\Local\Temp\ipykernel_20172\3917012578.py:19: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": query})


Answer:
 The PALMS process is about adapting language models to society by crafting and fine-tuning them on a dataset that reflects a predetermined set of target values. It aims to significantly change model behavior to address harmful and biased outputs and undesirable behavior according to specific cultural contexts. The process involves iterative steps, such as selecting topics, describing desired behavior, creating dataset prompts, and evaluating the model's performance using various metrics.

Sources:
- 3
Methodology
Figure 1: PALMS Steps
3.1
Step 1: Topic Selection
Choose a set of topics on which to adjust and improve model behavior.
We crafted a
list of what we considered sensitive topics (see Appe...
- Process for Adapting Language Models to
Society (PALMS) with Values-Targeted
Datasets
Irene Solaiman∗
Zillow Group
contact@irenesolaiman.com
Christy Dennison∗
MIT
christy@mit.edu
Abstract
Language mod...
- The co-occurrence evaluations analyze the most common word associated with

In [ ]:
while True:
    user_q = input("\nAsk a question (or type 'exit' to quit): ")
    if user_q.lower() in ["exit", "quit"]:
        break
    
    result = qa_chain({"query": user_q})
    print("\nAnswer:\n", result["result"])
    
    print("\nSources:")
    for i, doc in enumerate(result["source_documents"], start=1):
        print(f"Source {i}: {doc.page_content[:200]}...")



Ask a question (or type 'exit' to quit):  How is toxicity measured in PALMS?



Answer:
 Toxicity in PALMS is measured using two main methods: toxicity scoring and human evaluations. The toxicity scoring method uses the Perspective API to analyze model outputs for toxicity. Human evaluators also rate how well the model outputs conform to a predetermined set of values, which includes assessing toxicity. These evaluations help determine the level of toxicity in the model's language output.

Sources:
Source 1: PALMS to their models. Evaluating alignment and harmful outputs cannot be done by any
one metric and means of evaluation is a constantly growing ﬁeld of research. Quantitative
evaluations especially a...
Source 2: targeted model and base model, conﬁrming that high-quality data can help improve toxicity,
but not nearly as eﬃciently as from ﬁne-tuning on a values-targeted dataset constructed
with PALMS. See Appen...
Source 3: few samples. We refer to the models ﬁne-tuned using PALMS as values-targeted models and
the dataset used to train that model as the values


Ask a question (or type 'exit' to quit):  What are the key goals of PALMS?



Answer:
 The key goals of PALMS (Predictive Assessment of Language Model Sensitivity) include adjusting and improving model behavior on sensitive topics, guiding the language model to exhibit desired behaviors on specific topics, and creating values-targeted dataset samples to influence the model's outputs positively.

Sources:
Source 1: The co-occurrence evaluations analyze the most common word associated with a given social
category and make qualitative comparisons between the models. PALMS is iterative, and
training dataset example...
Source 2: nity around accountability, scaling laws, generalizability, and other generative models. See
Appendix N for questions.
8
7
Limitations
This research was only conducted in the U.S. English language and...
Source 3: 3
Methodology
Figure 1: PALMS Steps
3.1
Step 1: Topic Selection
Choose a set of topics on which to adjust and improve model behavior.
We crafted a
list of what we considered sensitive topics (see Appe...



Ask a question (or type 'exit' to quit):  How does PALMS handle sensitive topics?



Answer:
 PALMS handles sensitive topics by following a specific methodology that includes steps such as topic selection, desired behavior description, dataset prompt creation, and co-occurrence evaluations. The approach involves crafting position statements for chosen categories, writing prompts for the language model, and analyzing the most common words associated with social categories to make qualitative comparisons between models. PALMS is iterative, allowing for the addition of training dataset examples depending on validation set performance. It aims to improve model behavior on sensitive topics by guiding the language model towards desired behaviors through a structured process.

Sources:
Source 1: 3
Methodology
Figure 1: PALMS Steps
3.1
Step 1: Topic Selection
Choose a set of topics on which to adjust and improve model behavior.
We crafted a
list of what we considered sensitive topics (see Appe...
Source 2: The co-occurrence evaluations analyze the most common word associated 


Ask a question (or type 'exit' to quit):  How does PALMS adapt models to specific cultural contexts



Answer:
 PALMS adapts models to specific cultural contexts by crafting and fine-tuning on datasets that reflect predetermined sets of target values. The process involves iteratively changing model behavior by adding training dataset examples based on observed shortcomings from evaluations. By evaluating outputs for adherence to target values, toxicity scoring, and analyzing common words associated with social categories, PALMS aims to significantly improve model behavior to align with specific cultural contexts. However, it is important to note that PALMS was developed and evaluated primarily within the U.S. English language and may have difficulty replicating underrepresented cultural contexts. Additionally, the positions and values used in PALMS are influenced by a U.S. lens, U.S. law, and industry priorities, which may not align with all cultural perspectives.

Sources:
Source 1: Process for Adapting Language Models to
Society (PALMS) with Values-Targeted
Datasets
Irene Solaiman∗
Z